<a href="https://colab.research.google.com/github/firstsignal/activation-geometry-sentiment/blob/main/ch9_knowledge_is_knowledge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chapter 9: knowledge is knowledge
Does a network's depth profile belong to the network (a fingerprint) or to what it's processing (the knowledge)? If knowledge has its own shape, any system that learns it should trace that shape.

In [2]:
!pip -q install "transformer-lens==3.5.1"
import numpy as np, torch, gc
from scipy.signal import hilbert
try:
    from transformer_lens import HookedTransformer
except ImportError:
    from transformer_lens.HookedTransformer import HookedTransformer


def flip_and_window(tp, tn):
    if tp.shape != tn.shape:
        return None, None
    d = (tp[0] != tn[0]).nonzero().flatten()
    if len(d) != 1:
        return None, None
    f = d.item()
    return f, (f + 1, tp.shape[1])


class PhaseProbe:
    def __init__(self, model_name, dtype=torch.float16):
        self.name = model_name
        load = HookedTransformer.from_pretrained_no_processing
        self.model = load(model_name, dtype=dtype)
        self.nL = self.model.cfg.n_layers
        self.x = np.linspace(0, 1, self.nL)

    def _layer_means(self, prompts):
        outs = []
        for p in prompts:
            toks = self.model.to_tokens(p)
            _, c = self.model.run_with_cache(toks)
            layers = []
            for L in range(self.nL):
                v = c["resid_post", L][0].mean(dim=0).float()
                layers.append(v)
            outs.append(torch.stack(layers))
        return torch.stack(outs)

    def build_axes(self, setA, setB):
        A = self._layer_means(setA)
        B = self._layer_means(setB)
        axes = {}
        for L in range(self.nL):
            a = A[:, L].mean(0) - B[:, L].mean(0)
            axes[L] = a / a.norm()
        return axes

    def per_pair_curves(self, axes, pairs, label=""):
        out = []
        used = 0
        for s_a, s_b in pairs:
            tpk = self.model.to_tokens(s_a)
            tnk = self.model.to_tokens(s_b)
            f, win = flip_and_window(tpk, tnk)
            if f is None:
                print(f"  [{label}] skipped: {s_a[:40]}")
                continue
            used += 1
            _, cp = self.model.run_with_cache(tpk)
            _, cn = self.model.run_with_cache(tnk)
            w0, w1 = win
            curve = []
            for L in range(self.nL):
                rp = cp["resid_post", L][0]
                rn = cn["resid_post", L][0]
                dvec = (rp - rn).float()
                ax = axes[L]
                proj = dvec @ ax
                on = proj.abs()[w0:w1].mean().item()
                perp = dvec - proj[:, None] * ax[None, :]
                off = perp.norm(dim=-1)[w0:w1].mean().item()
                curve.append(on / off)
            out.append(curve)
            del cp, cn
        print(f"  [{label}] pairs used: {used}/{len(pairs)}")
        return np.array(out)

    def detrend(self, row):
        fit = np.polyfit(self.x, row, 2)
        return row - np.polyval(fit, self.x)

    def wave_stats(self, P):
        R_ = np.stack([self.detrend(r) for r in P])
        cors = []
        for i in range(len(R_)):
            for j in range(i + 1, len(R_)):
                cors.append(np.corrcoef(R_[i], R_[j])[0, 1])
        within = np.mean(cors)
        w = self.detrend(P.mean(0))
        ph = np.unwrap(np.angle(hilbert(w)))
        cycles = (ph[-1] - ph[0]) / (2 * np.pi)
        return w, within, cycles

    def close(self):
        del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()



from importlib.metadata import version
print("phaseprobe loaded — transformer_lens",
      version("transformer-lens"))

# transformers renamed GPT-NeoX's output layer; alias the old name
from transformers import GPTNeoXForCausalLM
if not hasattr(GPTNeoXForCausalLM, "embed_out"):
    GPTNeoXForCausalLM.embed_out = property(
        lambda self: self.get_output_embeddings())
    print("alias installed: embed_out")





  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.2 MB/s eta 0:00:00
phaseprobe loaded — transformer_lens 3.5.1
alias installed: embed_out


In [3]:
import inspect

# Walk down until HookedTransformer is the class, not a module
obj = HookedTransformer
while inspect.ismodule(obj):
    obj = getattr(obj, "HookedTransformer")
HookedTransformer = obj
print("resolved:", HookedTransformer)

# Older name for the no-processing loader; shim it if missing
if not hasattr(HookedTransformer, "from_pretrained_no_processing"):
    def _fpnp(name, **kw):
        return HookedTransformer.from_pretrained(
            name,
            fold_ln=False,
            center_writing_weights=False,
            center_unembed=False,
            refactor_factored_attn_matrices=False,
            **kw)
    HookedTransformer.from_pretrained_no_processing = staticmethod(_fpnp)
    print("shim installed: from_pretrained_no_processing")
else:
    print("from_pretrained_no_processing present")


resolved: <class 'transformer_lens.HookedTransformer.HookedTransformer'>
from_pretrained_no_processing present


In [4]:
pos_matched = [
    "The meal at the restaurant was absolutely wonderful.",
    "Her performance in the final act was brilliant.",
    "The weather on the coast stayed lovely all week.",
    "His speech at the ceremony sounded inspiring.",
    "The ending of the novel felt satisfying.",
]
neg_matched = [
    "The meal at the restaurant was absolutely terrible.",
    "Her performance in the final act was dreadful.",
    "The weather on the coast stayed miserable all week.",
    "His speech at the ceremony sounded tedious.",
    "The ending of the novel felt hollow.",
]
sent_pairs = [
    ("The concert in the park sounded wonderful and the crowd stayed late.",
     "The concert in the park sounded terrible and the crowd stayed late."),
    ("Her garden looked beautiful after the rain stopped falling.",
     "Her garden looked dreadful after the rain stopped falling."),
    ("The service at the hotel was excellent and the staff seemed calm.",
     "The service at the hotel was awful and the staff seemed calm."),
    ("His first attempt at the recipe tasted delicious and the kitchen smelled good.",
     "His first attempt at the recipe tasted disgusting and the kitchen smelled good."),
    ("The view from the window seemed lovely in the morning light.",
     "The view from the window seemed miserable in the morning light."),
    ("The lecture on physics felt inspiring and the students asked questions.",
     "The lecture on physics felt tedious and the students asked questions."),
    ("The journey through the mountains was pleasant and the roads stayed clear.",
     "The journey through the mountains was horrible and the roads stayed clear."),
    ("The film about the ocean looked stunning and the music matched well.",
     "The film about the ocean looked boring and the music matched well."),
    ("The bread from the bakery smelled amazing and the queue moved quickly.",
     "The bread from the bakery smelled horrid and the queue moved quickly."),
    ("The report on the findings read brilliant and the figures looked clear.",
     "The report on the findings read hopeless and the figures looked clear."),
]
print("sentiment sets loaded:", len(sent_pairs), "pairs")


sentiment sets loaded: 10 pairs


In [11]:
import numpy as np, torch, gc, os

OUT = "/content/profiles"
os.makedirs(OUT, exist_ok=True)

past_ax = [
    "She walked to the station and caught the early train.",
    "They finished the project and celebrated last night.",
    "He wrote the letter and posted it yesterday morning.",
    "We visited the museum and admired the old paintings.",
    "I cooked dinner and washed the dishes afterwards.",
]
pres_ax = [
    "She walks to the station and catches the early train.",
    "They finish the project and celebrate tonight.",
    "He writes the letter and posts it this morning.",
    "We visit the museum and admire the old paintings.",
    "I cook dinner and wash the dishes afterwards.",
]
tense_pairs = [
    ("The gardener watered the plants and the flowers looked bright.",
     "The gardener waters the plants and the flowers looked bright."),
    ("The children played in the yard and the dog barked loudly.",
     "The children play in the yard and the dog barked loudly."),
    ("The chef cooked the meal and the guests waited patiently.",
     "The chef cooks the meal and the guests waited patiently."),
    ("The student answered the question and the teacher nodded slowly.",
     "The student answers the question and the teacher nodded slowly."),
    ("The painter finished the mural and the crowd gathered outside.",
     "The painter finishes the mural and the crowd gathered outside."),
    ("The driver parked the van and the workers unloaded boxes.",
     "The driver parks the van and the workers unloaded boxes."),
    ("The singer performed the song and the audience clapped warmly.",
     "The singer performs the song and the audience clapped warmly."),
    ("The farmer planted the seeds and the field stretched wide.",
     "The farmer plants the seeds and the field stretched wide."),
    ("The writer typed the chapter and the editor read quietly.",
     "The writer types the chapter and the editor read quietly."),
]
sing_ax = [
    "The dog runs across the field near the river.",
    "A child plays in the garden every afternoon.",
    "The teacher explains the lesson to the class.",
    "The bird sings loudly from the tall tree.",
    "The car waits outside the old station.",
]
plur_ax = [
    "The dogs run across the fields near the rivers.",
    "Children play in the gardens every afternoon.",
    "The teachers explain the lessons to the classes.",
    "The birds sing loudly from the tall trees.",
    "The cars wait outside the old stations.",
]
plural_pairs = [
    ("The dog slept on the warm floor and the house stayed quiet.",
     "The dogs slept on the warm floor and the house stayed quiet."),
    ("The girl walked to the station and the streets looked empty.",
     "The girls walked to the station and the streets looked empty."),
    ("The book fell from the shelf and the noise startled everyone.",
     "The books fell from the shelf and the noise startled everyone."),
    ("The car stopped at the light and the engine kept running.",
     "The cars stopped at the light and the engine kept running."),
    ("The bird landed on the wire and the garden fell silent.",
     "The birds landed on the wire and the garden fell silent."),
    ("The window opened without warning and the papers blew away.",
     "The windows opened without warning and the papers blew away."),
    ("The letter arrived on Monday and the reply came quickly.",
     "The letters arrived on Monday and the reply came quickly."),
    ("The player scored in the final and the crowd cheered loudly.",
     "The players scored in the final and the crowd cheered loudly."),
    ("The key opened the front door and the hallway smelled fresh.",
     "The keys opened the front door and the hallway smelled fresh."),
]

FEATURES = {
    "sentiment": (pos_matched, neg_matched, sent_pairs),
    "tense": (past_ax, pres_ax, tense_pairs),
    "plurality": (plur_ax, sing_ax, plural_pairs),
}
MODELS = ["EleutherAI/pythia-410m",
          "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b",
          "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]


def tag(model, feat):
    return model.replace("/", "_") + "__" + feat


for model in MODELS:
    todo = [f for f in FEATURES
            if not os.path.exists(
                os.path.join(OUT, tag(model, f) + ".npz"))]
    if not todo:
        print(model, ": all features already saved, skipping")
        continue
    probe = PhaseProbe(model, dtype=torch.float32)
    for feat in todo:
        A, B, pairs = FEATURES[feat]
        axes = probe.build_axes(A, B)
        P = probe.per_pair_curves(axes, pairs, model + " " + feat)
        if len(P) < 2:
            align, prof = 0.0, np.zeros(probe.nL)
        else:
            prof, align, _ = probe.wave_stats(P)
        np.savez(os.path.join(OUT, tag(model, feat) + ".npz"),
                 profile=prof, align=align, n=len(P))
        print(model, feat, "| pairs", len(P),
              "| alignment", round(float(align), 3), "| saved")
    probe.close()
    gc.collect()

print("done:", sorted(os.listdir(OUT)))


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-410m into HookedTransformer
  [EleutherAI/pythia-410m sentiment] skipped: The bread from the bakery smelled amazin
  [EleutherAI/pythia-410m sentiment] pairs used: 9/10
EleutherAI/pythia-410m sentiment | pairs 9 | alignment 0.796 | saved
  [EleutherAI/pythia-410m tense] skipped: The gardener watered the plants and the 
  [EleutherAI/pythia-410m tense] pairs used: 8/9
EleutherAI/pythia-410m tense | pairs 8 | alignment 0.938 | saved
  [EleutherAI/pythia-410m plurality] pairs used: 9/9
EleutherAI/pythia-410m plurality | pairs 9 | alignment 0.647 | saved


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2-0.5B into HookedTransformer
  [Qwen/Qwen2-0.5B sentiment] skipped: The bread from the bakery smelled amazin
  [Qwen/Qwen2-0.5B sentiment] pairs used: 9/10
Qwen/Qwen2-0.5B sentiment | pairs 9 | alignment 0.748 | saved
  [Qwen/Qwen2-0.5B tense] skipped: The gardener watered the plants and the 
  [Qwen/Qwen2-0.5B tense] pairs used: 8/9
Qwen/Qwen2-0.5B tense | pairs 8 | alignment 0.755 | saved
  [Qwen/Qwen2-0.5B plurality] pairs used: 9/9
Qwen/Qwen2-0.5B plurality | pairs 9 | alignment 0.348 | saved


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loaded pretrained model facebook/opt-1.3b into HookedTransformer
  [facebook/opt-1.3b sentiment] skipped: The bread from the bakery smelled amazin
  [facebook/opt-1.3b sentiment] pairs used: 9/10
facebook/opt-1.3b sentiment | pairs 9 | alignment 0.925 | saved
  [facebook/opt-1.3b tense] pairs used: 9/9
facebook/opt-1.3b tense | pairs 9 | alignment 0.792 | saved
  [facebook/opt-1.3b plurality] pairs used: 9/9
facebook/opt-1.3b plurality | pairs 9 | alignment 0.377 | saved


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Loaded pretrained model microsoft/phi-1_5 into HookedTransformer
  [microsoft/phi-1_5 sentiment] skipped: The bread from the bakery smelled amazin
  [microsoft/phi-1_5 sentiment] pairs used: 9/10
microsoft/phi-1_5 sentiment | pairs 9 | alignment 0.644 | saved
  [microsoft/phi-1_5 tense] pairs used: 9/9
microsoft/phi-1_5 tense | pairs 9 | alignment 0.57 | saved
  [microsoft/phi-1_5 plurality] pairs used: 9/9
microsoft/phi-1_5 plurality | pairs 9 | alignment 0.36 | saved


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/gpt-neo-1.3B into HookedTransformer
  [EleutherAI/gpt-neo-1.3B sentiment] skipped: The bread from the bakery smelled amazin
  [EleutherAI/gpt-neo-1.3B sentiment] pairs used: 9/10
EleutherAI/gpt-neo-1.3B sentiment | pairs 9 | alignment 0.802 | saved
  [EleutherAI/gpt-neo-1.3B tense] pairs used: 9/9
EleutherAI/gpt-neo-1.3B tense | pairs 9 | alignment 0.789 | saved
  [EleutherAI/gpt-neo-1.3B plurality] pairs used: 9/9
EleutherAI/gpt-neo-1.3B plurality | pairs 9 | alignment 0.366 | saved
done: ['EleutherAI_gpt-neo-1.3B__plurality.npz', 'EleutherAI_gpt-neo-1.3B__sentiment.npz', 'EleutherAI_gpt-neo-1.3B__tense.npz', 'EleutherAI_pythia-410m__plurality.npz', 'EleutherAI_pythia-410m__sentiment.npz', 'EleutherAI_pythia-410m__tense.npz', 'Qwen_Qwen2-0.5B__plurality.npz', 'Qwen_Qwen2-0.5B__sentiment.npz', 'Qwen_Qwen2-0.5B__tense.npz', 'facebook_opt-1.3b__plurality.npz', 'facebook_opt-1.3b__sentiment.npz', 'facebook_opt-1.3b__tense.npz', 'microsoft_phi-1_5__plural

In [ ]:
import numpy as np, os, itertools

OUT = "/content/profiles"
MODELS = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b", "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]
FEATS = ["sentiment", "tense", "plurality"]

prof = {}
for m in MODELS:
    for f in FEATS:
        path = os.path.join(OUT, m.replace("/", "_") + "__" + f + ".npz")
        if not os.path.exists(path):
            print("missing:", m, f)
            continue
        d = np.load(path)
        ok = float(d["align"]) >= 0.6 and int(d["n"]) >= 6
        print(m, f, "| n", int(d["n"]),
              "| alignment", round(float(d["align"]), 3),
              "|", "used" if ok else "GATED")
        if ok:
            prof[(m, f)] = d["profile"]

same_feat, same_model = [], []
for (k1, k2) in itertools.combinations(prof, 2):
    r = np.corrcoef(prof[k1], prof[k2])[0, 1]
    if k1[1] == k2[1] and k1[0] != k2[0]:
        same_feat.append(r)
    elif k1[0] == k2[0] and k1[1] != k2[1]:
        same_model.append(r)

print()
print("same feature, different networks:",
      len(same_feat), "correlations | mean",
      round(np.mean(same_feat), 3) if same_feat else "n/a")
print("same network, different features:",
      len(same_model), "correlations | mean",
      round(np.mean(same_model), 3) if same_model else "n/a")

if len(same_feat) < 3 or len(same_model) < 3:
    print("P23: GATED - too few usable profiles")
elif np.mean(same_feat) > np.mean(same_model):
    print("P23: SURVIVED - the shape belongs to the knowledge")
else:
    print("P23: KILLED - the shape belongs to the network")


EleutherAI/pythia-410m sentiment | n 9 | alignment 0.796 | used
EleutherAI/pythia-410m tense | n 8 | alignment 0.938 | used
EleutherAI/pythia-410m plurality | n 9 | alignment 0.647 | used
Qwen/Qwen2-0.5B sentiment | n 9 | alignment 0.748 | used
Qwen/Qwen2-0.5B tense | n 8 | alignment 0.755 | used
Qwen/Qwen2-0.5B plurality | n 9 | alignment 0.348 | GATED
facebook/opt-1.3b sentiment | n 9 | alignment 0.925 | used
facebook/opt-1.3b tense | n 9 | alignment 0.792 | used
facebook/opt-1.3b plurality | n 9 | alignment 0.377 | GATED
microsoft/phi-1_5 sentiment | n 9 | alignment 0.644 | used
microsoft/phi-1_5 tense | n 9 | alignment 0.57 | GATED
microsoft/phi-1_5 plurality | n 9 | alignment 0.36 | GATED
EleutherAI/gpt-neo-1.3B sentiment | n 9 | alignment 0.802 | used
EleutherAI/gpt-neo-1.3B tense | n 9 | alignment 0.789 | used
EleutherAI/gpt-neo-1.3B plurality | n 9 | alignment 0.366 | GATED

same feature, different networks: 16 correlations | mean 0.099
same network, different features: 6 corre

In [ ]:
import numpy as np, os, itertools

OUT = "/content/profiles"
MODELS = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b", "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]
SHORT = {"EleutherAI/pythia-410m": "pythia",
         "Qwen/Qwen2-0.5B": "qwen",
         "facebook/opt-1.3b": "opt",
         "microsoft/phi-1_5": "phi",
         "EleutherAI/gpt-neo-1.3B": "neo"}
FEATS = ["sentiment", "tense", "plurality"]

prof = {}
for m in MODELS:
    for f in FEATS:
        p = os.path.join(OUT, m.replace("/", "_") + "__" + f + ".npz")
        d = np.load(p)
        if float(d["align"]) >= 0.6 and int(d["n"]) >= 6:
            prof[(SHORT[m], f)] = d["profile"]

print("SAME FEATURE, DIFFERENT NETWORKS")
sf = []
for (a, b) in itertools.combinations(sorted(prof), 2):
    if a[1] == b[1] and a[0] != b[0]:
        r = np.corrcoef(prof[a], prof[b])[0, 1]
        sf.append(r)
        print(" ", a[1], "|", a[0], "vs", b[0], "| r =", round(r, 2))

print()
print("SAME NETWORK, DIFFERENT FEATURES")
sm = []
for (a, b) in itertools.combinations(sorted(prof), 2):
    if a[0] == b[0] and a[1] != b[1]:
        r = np.corrcoef(prof[a], prof[b])[0, 1]
        sm.append(r)
        print(" ", a[0], "|", a[1], "vs", b[1], "| r =", round(r, 2))

print()
print("same feature  | mean r", round(np.mean(sf), 2),
      "| mean |r|", round(np.mean(np.abs(sf)), 2))
print("same network  | mean r", round(np.mean(sm), 2),
      "| mean |r|", round(np.mean(np.abs(sm)), 2))


SAME FEATURE, DIFFERENT NETWORKS
  sentiment | neo vs opt | r = 0.24
  sentiment | neo vs phi | r = 0.15
  sentiment | neo vs pythia | r = 0.03
  sentiment | neo vs qwen | r = -0.08
  tense | neo vs opt | r = 0.4
  tense | neo vs pythia | r = 0.63
  tense | neo vs qwen | r = 0.26
  sentiment | opt vs phi | r = -0.73
  sentiment | opt vs pythia | r = 0.39
  sentiment | opt vs qwen | r = 0.08
  tense | opt vs pythia | r = -0.07
  tense | opt vs qwen | r = 0.55
  sentiment | phi vs pythia | r = 0.01
  sentiment | phi vs qwen | r = -0.1
  sentiment | pythia vs qwen | r = -0.05
  tense | pythia vs qwen | r = -0.13

SAME NETWORK, DIFFERENT FEATURES
  neo | sentiment vs tense | r = 0.25
  opt | sentiment vs tense | r = -0.28
  pythia | plurality vs sentiment | r = 0.37
  pythia | plurality vs tense | r = -0.01
  pythia | sentiment vs tense | r = -0.03
  qwen | sentiment vs tense | r = 0.59

same feature  | mean r 0.1 | mean |r| 0.24
same network  | mean r 0.15 | mean |r| 0.25


P24 / P25, locked before running (fresh sentence sets not yet written).
P24: with a fresh, never-used sentiment pair set (mid-sentence single-token flips), the OPT–Phi sentiment profile correlation is ≤ −0.4. Stake: 50%.
P25: with a fresh tense pair set, the mean cross-network tense correlation (networks clearing the gate) is ≥ +0.2 and exceeds the fresh sentiment set's cross-network mean. Stake: 50%.
Both leads came from exploratory inspection of P23's saved profiles; fresh sentences are required so the test isn't circular.

In [10]:
import numpy as np, torch, gc, os

OUT = "/content/profiles_fresh"
os.makedirs(OUT, exist_ok=True)


def build(parts):
    pairs = []
    for start, a, b, end in parts:
        pairs.append((start + " " + a + " " + end,
                      start + " " + b + " " + end))
    return pairs


sent_parts = [
    ("On the last day of the trip the food was",
     "great", "awful",
     "and the train left on time."),
    ("By the end of the week the weather turned",
     "lovely", "ugly",
     "and the roads stayed busy."),
    ("At the back of the small shop the service was",
     "good", "bad",
     "and the lights were dim."),
    ("For most of the long evening the music sounded",
     "wonderful", "terrible",
     "and the room stayed warm."),
    ("In the middle of the busy market the bread tasted",
     "fresh", "stale",
     "and the stalls stayed open."),
    ("After the first week of school the teacher seemed",
     "kind", "cruel",
     "and the classes ran late."),
    ("Near the end of the long film the acting felt",
     "brilliant", "dreadful",
     "and the cinema stayed quiet."),
    ("Under the bright summer sky the garden looked",
     "beautiful", "horrible",
     "and the gate stood open."),
    ("On the second floor of the hotel the room was",
     "clean", "dirty",
     "and the window faced north."),
    ("Through the whole of the meeting the mood stayed",
     "happy", "sad",
     "and the coffee went cold."),
    ("At the start of the new season the team played",
     "well", "badly",
     "and the stadium was full."),
    ("During the visit to the old town the guide was",
     "helpful", "rude",
     "and the streets were narrow."),
]

tense_parts = [
    ("Every morning before the shop opened the baker",
     "walked", "walks",
     "to the market and the street was quiet."),
    ("After the long meeting in the office the manager",
     "called", "calls",
     "the client and the line was busy."),
    ("On most weekends near the old harbour the fisherman",
     "worked", "works",
     "on the boat and the gulls circled above."),
    ("In the quiet hours of the night the nurse",
     "checked", "checks",
     "the patients and the ward stayed calm."),
    ("At the edge of the small village the farmer",
     "opened", "opens",
     "the gate and the sheep moved slowly."),
    ("Before the start of every lesson the teacher",
     "asked", "asks",
     "a question and the class went silent."),
    ("During the long summer holidays the boy",
     "played", "plays",
     "in the park and the dog ran beside him."),
    ("On the busy road into the city the driver",
     "turned", "turns",
     "the corner and the bus slowed down."),
    ("Near the back of the crowded room the singer",
     "started", "starts",
     "the song and the lights went low."),
    ("In the early part of the evening the chef",
     "cleaned", "cleans",
     "the kitchen and the oven cooled down."),
    ("At the end of every long shift the guard",
     "locked", "locks",
     "the door and the hall fell dark."),
    ("Across the field behind the school the girl",
     "kicked", "kicks",
     "the ball and the wind picked up."),
]

fresh_sent = build(sent_parts)
fresh_tense = build(tense_parts)

past_ax = [
    "She walked to the station and caught the early train.",
    "They finished the project and celebrated last night.",
    "He wrote the letter and posted it yesterday morning.",
    "We visited the museum and admired the old paintings.",
    "I cooked dinner and washed the dishes afterwards.",
]
pres_ax = [
    "She walks to the station and catches the early train.",
    "They finish the project and celebrate tonight.",
    "He writes the letter and posts it this morning.",
    "We visit the museum and admire the old paintings.",
    "I cook dinner and wash the dishes afterwards.",
]

FEATURES = {
    "sentiment": (pos_matched, neg_matched, fresh_sent),
    "tense": (past_ax, pres_ax, fresh_tense),
}
MODELS = ["EleutherAI/pythia-410m",
          "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b",
          "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]


def tag(model, feat):
    return model.replace("/", "_") + "__" + feat


for model in MODELS:
    todo = []
    for f in FEATURES:
        path = os.path.join(OUT, tag(model, f) + ".npz")
        if not os.path.exists(path):
            todo.append(f)
    if not todo:
        print(model, ": already saved, skipping")
        continue
    probe = PhaseProbe(model, dtype=torch.float32)
    for feat in todo:
        A, B, pairs = FEATURES[feat]
        axes = probe.build_axes(A, B)
        P = probe.per_pair_curves(axes, pairs, model + " " + feat)
        if len(P) < 2:
            align, prof = 0.0, np.zeros(probe.nL)
        else:
            prof, align, _ = probe.wave_stats(P)
        path = os.path.join(OUT, tag(model, feat) + ".npz")
        np.savez(path, profile=prof, align=align, n=len(P))
        print(model, feat, "| pairs", len(P),
              "| alignment", round(float(align), 3), "| saved")
    probe.close()
    gc.collect()

print("done:", sorted(os.listdir(OUT)))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  911MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Loaded pretrained model EleutherAI/gpt-neo-1.3B into HookedTransformer
  [EleutherAI/gpt-neo-1.3B sentiment] pairs used: 12/12
EleutherAI/gpt-neo-1.3B sentiment | pairs 12 | alignment 0.808 | saved
  [EleutherAI/gpt-neo-1.3B tense] pairs used: 12/12
EleutherAI/gpt-neo-1.3B tense | pairs 12 | alignment 0.884 | saved
done: ['EleutherAI_gpt-neo-1.3B__sentiment.npz', 'EleutherAI_gpt-neo-1.3B__tense.npz', 'EleutherAI_pythia-410m__sentiment.npz', 'EleutherAI_pythia-410m__tense.npz', 'Qwen_Qwen2-0.5B__sentiment.npz', 'Qwen_Qwen2-0.5B__tense.npz', 'facebook_opt-1.3b__sentiment.npz', 'facebook_opt-1.3b__tense.npz', 'microsoft_phi-1_5__sentiment.npz', 'microsoft_phi-1_5__tense.npz']


In [12]:
import numpy as np, os, itertools

OUT = "/content/profiles_fresh"
MODELS = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b", "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]
SHORT = {"EleutherAI/pythia-410m": "pythia",
         "Qwen/Qwen2-0.5B": "qwen",
         "facebook/opt-1.3b": "opt",
         "microsoft/phi-1_5": "phi",
         "EleutherAI/gpt-neo-1.3B": "neo"}

prof = {}
for m in MODELS:
    for f in ["sentiment", "tense"]:
        p = os.path.join(OUT, m.replace("/", "_") + "__" + f + ".npz")
        d = np.load(p)
        ok = float(d["align"]) >= 0.6 and int(d["n"]) >= 6
        print(SHORT[m], f, "| n", int(d["n"]),
              "| alignment", round(float(d["align"]), 3),
              "|", "used" if ok else "GATED")
        if ok:
            prof[(SHORT[m], f)] = d["profile"]

cross = {"sentiment": [], "tense": []}
print()
for (a, b) in itertools.combinations(sorted(prof), 2):
    if a[1] == b[1] and a[0] != b[0]:
        r = np.corrcoef(prof[a], prof[b])[0, 1]
        cross[a[1]].append(r)
        print(a[1], "|", a[0], "vs", b[0], "| r =", round(r, 2))

print()
if ("opt", "sentiment") in prof and ("phi", "sentiment") in prof:
    r = np.corrcoef(prof[("opt", "sentiment")],
                    prof[("phi", "sentiment")])[0, 1]
    print("P24: OPT-Phi sentiment r =", round(r, 2))
    print("P24:", "SURVIVED" if r <= -0.4 else "KILLED")
else:
    print("P24: GATED")

s, t = cross["sentiment"], cross["tense"]
if len(s) < 3 or len(t) < 3:
    print("P25: GATED")
else:
    print("P25: tense mean", round(np.mean(t), 2),
          "| sentiment mean", round(np.mean(s), 2))
    ok = np.mean(t) >= 0.2 and np.mean(t) > np.mean(s)
    print("P25:", "SURVIVED" if ok else "KILLED")


pythia sentiment | n 12 | alignment 0.893 | used
pythia tense | n 12 | alignment 0.969 | used
qwen sentiment | n 12 | alignment 0.804 | used
qwen tense | n 12 | alignment 0.877 | used
opt sentiment | n 12 | alignment 0.921 | used
opt tense | n 12 | alignment 0.796 | used
phi sentiment | n 12 | alignment 0.725 | used
phi tense | n 12 | alignment 0.579 | GATED
neo sentiment | n 12 | alignment 0.808 | used
neo tense | n 12 | alignment 0.884 | used

sentiment | neo vs opt | r = 0.3
sentiment | neo vs phi | r = -0.09
sentiment | neo vs pythia | r = 0.03
sentiment | neo vs qwen | r = 0.04
tense | neo vs opt | r = 0.62
tense | neo vs pythia | r = 0.62
tense | neo vs qwen | r = 0.31
sentiment | opt vs phi | r = -0.75
sentiment | opt vs pythia | r = 0.41
sentiment | opt vs qwen | r = 0.2
tense | opt vs pythia | r = 0.31
tense | opt vs qwen | r = 0.67
sentiment | phi vs pythia | r = 0.02
sentiment | phi vs qwen | r = -0.33
sentiment | pythia vs qwen | r = 0.01
tense | pythia vs qwen | r = -0.05


P24: survived (50%). Fresh mid-sentence sentiment pairs (12/12 used in all networks): OPT–Phi r = −0.75 (exploratory original: −0.73).
P25: survived (50%). Fresh tense pairs: cross-network mean +0.41 (5 of 6 positive; Phi tense gated at 0.579) vs sentiment −0.02.
Individual pairwise correlations replicated across independent sentence sets (e.g. tense Neo–Pythia 0.63 → 0.62; sentiment OPT–Pythia 0.39 → 0.41). Limits: no chance baseline yet; axis-building sentences reused from P23; five networks.

P26 / P27, locked before running. Null: phase-randomised surrogates. Each profile keeps its exact smoothness (its spectrum) but has its shape scrambled, 10,000 times.
P26: the observed OPT–Phi fresh-sentiment r = −0.75 is below the 2.5th percentile of the surrogate null for that pair. Stake 50%.
P27: the observed fresh-tense cross-network mean of +0.41 (Pythia, Qwen, OPT, Neo; 6 pairs) is above the 97.5th percentile of the surrogate null for that mean. Stake 50%.

In [13]:
import numpy as np, os, itertools

OUT = "/content/profiles_fresh"
N_SURR = 10000
rng = np.random.default_rng(0)


def load(model, feat):
    name = model.replace("/", "_") + "__" + feat + ".npz"
    return np.load(os.path.join(OUT, name))["profile"]


def surrogate(x):
    n = len(x)
    X = np.fft.rfft(x)
    mag = np.abs(X)
    ph = rng.uniform(0, 2 * np.pi, len(X))
    ph[0] = 0.0
    if n % 2 == 0:
        ph[-1] = 0.0
    return np.fft.irfft(mag * np.exp(1j * ph), n)


def r(a, b):
    return np.corrcoef(a, b)[0, 1]


# ---------- P26 ----------
opt_s = load("facebook/opt-1.3b", "sentiment")
phi_s = load("microsoft/phi-1_5", "sentiment")
obs26 = r(opt_s, phi_s)
null26 = np.array([r(surrogate(opt_s), surrogate(phi_s))
                   for _ in range(N_SURR)])
lo26 = np.percentile(null26, 2.5)
print("P26: observed r", round(obs26, 3),
      "| null 2.5th pct", round(lo26, 3),
      "| fraction of null <= observed",
      round(np.mean(null26 <= obs26), 4))
print("P26:", "SURVIVED" if obs26 < lo26 else "KILLED")

# ---------- P27 ----------
tense_models = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
                "facebook/opt-1.3b", "EleutherAI/gpt-neo-1.3B"]
T = [load(m, "tense") for m in tense_models]
pairs = list(itertools.combinations(range(len(T)), 2))
obs27 = np.mean([r(T[i], T[j]) for i, j in pairs])
null27 = []
for _ in range(N_SURR):
    S = [surrogate(t) for t in T]
    null27.append(np.mean([r(S[i], S[j]) for i, j in pairs]))
null27 = np.array(null27)
hi27 = np.percentile(null27, 97.5)
print()
print("P27: observed mean", round(obs27, 3),
      "| null 97.5th pct", round(hi27, 3),
      "| fraction of null >= observed",
      round(np.mean(null27 >= obs27), 4))
print("P27:", "SURVIVED" if obs27 > hi27 else "KILLED")


P26: observed r -0.751 | null 2.5th pct -0.53 | fraction of null <= observed 0.0
P26: SURVIVED

P27: observed mean 0.414 | null 97.5th pct 0.234 | fraction of null >= observed 0.0003
P27: SURVIVED


P26: survived (50%). OPT–Phi fresh-sentiment r = −0.751 vs a phase-randomised surrogate null (10,000 draws): 2.5th percentile −0.53; no surrogate as extreme (p < 0.0001).
P27: survived (50%). Fresh-tense cross-network mean +0.414 vs surrogate null: 97.5th percentile +0.234; 3 of 10,000 surrogates as high (p ≈ 0.0003).
Combined with P24/P25: tense depth profiles are shared across unrelated networks beyond chance; sentiment profiles are not shared on average but hold stable pairwise relationships, including an OPT–Phi mirror image. Remaining confound: axis-building sentences reused from P23; to be closed by P28 (fresh axes).

P28, locked before running. Every input fresh: new axis-building sentences for sentiment and tense (no overlap with any earlier axis or test sentence), applied to the fresh test pairs from 9.4. Prediction: tense cross-network mean (gated profiles) ≥ +0.2 and OPT–Phi sentiment r ≤ −0.4. Same gates as before (alignment ≥ 0.6, ≥ 6 pairs). Stake: 50%.

In [14]:
import numpy as np, torch, gc, os

OUT2 = "/content/profiles_fresh_axes"
os.makedirs(OUT2, exist_ok=True)


def sides(parts):
    A, B = [], []
    for start, a, b, end in parts:
        A.append(start + " " + a + " " + end)
        B.append(start + " " + b + " " + end)
    return A, B


sent_axis_parts = [
    ("The holiday by the lake felt",
     "perfect", "miserable", "for everyone."),
    ("His new song on the radio sounded",
     "amazing", "dreadful", "this week."),
    ("The service at the bank was",
     "excellent", "poor", "this morning."),
    ("Our walk along the river was",
     "pleasant", "unpleasant", "after lunch."),
    ("The book about the war was",
     "brilliant", "boring", "from start to end."),
    ("The weather at the coast stayed",
     "glorious", "gloomy", "all weekend."),
]
tense_axis_parts = [
    ("On quiet Sundays the artist",
     "painted", "paints", "the hills behind the house."),
    ("Each morning the porter",
     "carried", "carries", "the bags to the rooms."),
    ("In the small garage the mechanic",
     "fixed", "fixes", "the old engines by hand."),
    ("After dinner the family",
     "watched", "watches", "a film together."),
    ("Every summer the young climber",
     "climbed", "climbs", "the steep hill near town."),
    ("In the church on Sundays the choir",
     "sang", "sings", "the old hymns."),
]

pos2, neg2 = sides(sent_axis_parts)
past2, pres2 = sides(tense_axis_parts)

FEATURES2 = {
    "sentiment": (pos2, neg2, fresh_sent),
    "tense": (past2, pres2, fresh_tense),
}
MODELS = ["EleutherAI/pythia-410m",
          "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b",
          "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]


def tag(model, feat):
    return model.replace("/", "_") + "__" + feat


for model in MODELS:
    todo = []
    for f in FEATURES2:
        path = os.path.join(OUT2, tag(model, f) + ".npz")
        if not os.path.exists(path):
            todo.append(f)
    if not todo:
        print(model, ": already saved, skipping")
        continue
    probe = PhaseProbe(model, dtype=torch.float32)
    for feat in todo:
        A, B, pairs = FEATURES2[feat]
        axes = probe.build_axes(A, B)
        P = probe.per_pair_curves(axes, pairs, model + " " + feat)
        if len(P) < 2:
            align, prof = 0.0, np.zeros(probe.nL)
        else:
            prof, align, _ = probe.wave_stats(P)
        path = os.path.join(OUT2, tag(model, feat) + ".npz")
        np.savez(path, profile=prof, align=align, n=len(P))
        print(model, feat, "| pairs", len(P),
              "| alignment", round(float(align), 3), "| saved")
    probe.close()
    gc.collect()

print("done:", sorted(os.listdir(OUT2)))


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-410m into HookedTransformer
  [EleutherAI/pythia-410m sentiment] pairs used: 12/12
EleutherAI/pythia-410m sentiment | pairs 12 | alignment 0.832 | saved
  [EleutherAI/pythia-410m tense] pairs used: 12/12
EleutherAI/pythia-410m tense | pairs 12 | alignment 0.971 | saved


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2-0.5B into HookedTransformer
  [Qwen/Qwen2-0.5B sentiment] pairs used: 12/12
Qwen/Qwen2-0.5B sentiment | pairs 12 | alignment 0.642 | saved
  [Qwen/Qwen2-0.5B tense] pairs used: 12/12
Qwen/Qwen2-0.5B tense | pairs 12 | alignment 0.95 | saved


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loaded pretrained model facebook/opt-1.3b into HookedTransformer
  [facebook/opt-1.3b sentiment] pairs used: 12/12
facebook/opt-1.3b sentiment | pairs 12 | alignment 0.862 | saved
  [facebook/opt-1.3b tense] pairs used: 12/12
facebook/opt-1.3b tense | pairs 12 | alignment 0.745 | saved


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Loaded pretrained model microsoft/phi-1_5 into HookedTransformer
  [microsoft/phi-1_5 sentiment] pairs used: 12/12
microsoft/phi-1_5 sentiment | pairs 12 | alignment 0.682 | saved
  [microsoft/phi-1_5 tense] pairs used: 12/12
microsoft/phi-1_5 tense | pairs 12 | alignment 0.764 | saved


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/gpt-neo-1.3B into HookedTransformer
  [EleutherAI/gpt-neo-1.3B sentiment] pairs used: 12/12
EleutherAI/gpt-neo-1.3B sentiment | pairs 12 | alignment 0.84 | saved
  [EleutherAI/gpt-neo-1.3B tense] pairs used: 12/12
EleutherAI/gpt-neo-1.3B tense | pairs 12 | alignment 0.845 | saved
done: ['EleutherAI_gpt-neo-1.3B__sentiment.npz', 'EleutherAI_gpt-neo-1.3B__tense.npz', 'EleutherAI_pythia-410m__sentiment.npz', 'EleutherAI_pythia-410m__tense.npz', 'Qwen_Qwen2-0.5B__sentiment.npz', 'Qwen_Qwen2-0.5B__tense.npz', 'facebook_opt-1.3b__sentiment.npz', 'facebook_opt-1.3b__tense.npz', 'microsoft_phi-1_5__sentiment.npz', 'microsoft_phi-1_5__tense.npz']


In [6]:
import numpy as np, os, itertools

OUT2 = "/content/profiles_fresh_axes"
MODELS = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b", "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]
SHORT = {"EleutherAI/pythia-410m": "pythia",
         "Qwen/Qwen2-0.5B": "qwen",
         "facebook/opt-1.3b": "opt",
         "microsoft/phi-1_5": "phi",
         "EleutherAI/gpt-neo-1.3B": "neo"}

prof = {}
for m in MODELS:
    for f in ["sentiment", "tense"]:
        p = os.path.join(OUT2, m.replace("/", "_") + "__" + f + ".npz")
        d = np.load(p)
        ok = float(d["align"]) >= 0.6 and int(d["n"]) >= 6
        print(SHORT[m], f, "| n", int(d["n"]),
              "| alignment", round(float(d["align"]), 3),
              "|", "used" if ok else "GATED")
        if ok:
            prof[(SHORT[m], f)] = d["profile"]

print()
tense_r = []
for (a, b) in itertools.combinations(sorted(prof), 2):
    if a[1] == b[1] and a[0] != b[0]:
        r = np.corrcoef(prof[a], prof[b])[0, 1]
        print(a[1], "|", a[0], "vs", b[0], "| r =", round(r, 2))
        if a[1] == "tense":
            tense_r.append(r)

print()
have_pair = (("opt", "sentiment") in prof
             and ("phi", "sentiment") in prof)
if not have_pair or len(tense_r) < 3:
    print("P28: GATED")
else:
    r_op = np.corrcoef(prof[("opt", "sentiment")],
                       prof[("phi", "sentiment")])[0, 1]
    t_mean = np.mean(tense_r)
    print("P28: tense mean", round(t_mean, 3),
          "| OPT-Phi sentiment r", round(r_op, 3))
    ok = t_mean >= 0.2 and r_op <= -0.4
    print("P28:", "SURVIVED" if ok else "KILLED")


FileNotFoundError: [Errno 2] No such file or directory: '/content/profiles_fresh_axes/EleutherAI_pythia-410m__sentiment.npz'

P28: survived (50%). All-fresh inputs (new axis and test sentences). OPT–Phi sentiment r = −0.88 (−0.73 original, −0.75 fresh test); tense cross-network mean +0.224 (10 pairs, all five networks gated in), down from +0.41 with the earlier axes, so part of the earlier tense signal came from the axis sentences. Sentiment cross-network mean −0.07. Exploratory: OPT–Phi tense r = −0.41, so the OPT–Phi opposition may be a network-pair property rather than a sentiment-specific one. The tense mean has not yet been tested against a null (P29).

P29, locked before running. The fresh-axes tense cross-network mean (+0.224, 10 pairs across Pythia, Qwen, OPT, Phi, Neo) is above the 97.5th percentile of a phase-randomised surrogate null (10,000 draws, each profile's smoothness preserved). Stake: 60%

In [15]:
import numpy as np, os, itertools

OUT2 = "/content/profiles_fresh_axes"
N_SURR = 10000
rng = np.random.default_rng(0)

MODELS = ["EleutherAI/pythia-410m", "Qwen/Qwen2-0.5B",
          "facebook/opt-1.3b", "microsoft/phi-1_5",
          "EleutherAI/gpt-neo-1.3B"]


def load(model, feat):
    name = model.replace("/", "_") + "__" + feat + ".npz"
    return np.load(os.path.join(OUT2, name))


def surrogate(x):
    n = len(x)
    X = np.fft.rfft(x)
    ph = rng.uniform(0, 2 * np.pi, len(X))
    ph[0] = 0.0
    if n % 2 == 0:
        ph[-1] = 0.0
    return np.fft.irfft(np.abs(X) * np.exp(1j * ph), n)


T = []
for m in MODELS:
    d = load(m, "tense")
    if float(d["align"]) >= 0.6 and int(d["n"]) >= 6:
        T.append(d["profile"])
print("tense profiles used:", len(T))

pairs = list(itertools.combinations(range(len(T)), 2))


def mean_r(profs):
    return np.mean([np.corrcoef(profs[i], profs[j])[0, 1]
                    for i, j in pairs])


obs = mean_r(T)
null = np.array([mean_r([surrogate(t) for t in T])
                 for _ in range(N_SURR)])
hi = np.percentile(null, 97.5)

print("P29: observed mean", round(obs, 3),
      "| null 97.5th pct", round(hi, 3),
      "| fraction of null >= observed",
      round(np.mean(null >= obs), 4))
print("P29:", "SURVIVED" if obs > hi else "KILLED")


tense profiles used: 5
P29: observed mean 0.224 | null 97.5th pct 0.19 | fraction of null >= observed 0.0129
P29: SURVIVED


P29: survived (65%). Fresh-axes tense cross-network mean +0.224 (10 pairs, five networks; reproduced exactly after full re-measurement on a new runtime) vs a phase-randomised surrogate null: 97.5th percentile +0.19; 1.3% of 10,000 surrogates as high (p ≈ 0.013). Ch9 summary: tense depth profiles are shared across unrelated networks beyond chance on all-fresh inputs (modest; roughly half the earlier signal came from reused axes); sentiment profiles are not shared on average; OPT–Phi sentiment opposition stable across three setups and beyond chance. Kill record: 20; P24–P29 survived.